In [6]:
!pwd
!apt install nano -y
! sudo apt update


/kaggle/working
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
nano is already the newest version (6.2-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 139 not upgraded.
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:

In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
from huggingface_hub import snapshot_download
import os

# Define the target directory for inspection
dataset_path = "/kaggle/working/jaCappella"

# Download the repository
# We use snapshot_download to preserve the folder structure noted in your meta.csv
snapshot_download(
    repo_id="jaCappella/jaCappella", 
    repo_type="dataset", 
    local_dir=dataset_path,
    local_dir_use_symlinks=False
)

print(f"Dataset downloaded to: {dataset_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 608 files:   0%|          | 0/608 [00:00<?, ?it/s]

Dataset downloaded to: /kaggle/working/jaCappella


In [ ]:
# import subprocess

# repo_url = ""  # replace with the repo URL
# repo_path = "/kaggle/working/"
# repo_name = os.path.splitext(os.path.basename(repo_url))[0]
# target_dir = os.path.join(repo_path, repo_name) 

# subprocess.run(["git", "pull", "--branch", "dev", repo_url], check=True)

In [7]:
pip install -r /kaggle/working/SepACap/requirements.txt

  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2024.8.30-py3-none-any.whl.metadata (2.2 kB)
  Using cached cffi-1.17.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached charset_normalizer-3.4.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (34 kB)
  Using cached contourpy-1.3.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.4 kB)
  Using cached decorator-5.1.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached filelock-3.16.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached fonttools-4.54.1-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (163 kB)
  Using cached fsspec-2024.9.0-py3-none-any.whl.metadata (11 kB)
  Using cached grpcio-1.67.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.9 kB)
  Using cached idna-3.1

In [8]:
pip install thop ptflops mir_eval nano torch loguru torchvision torchaudio librosa torchinfo

Note: you may need to restart the kernel to use updated packages.


In [9]:
!pwd
!ls
!ls jaCappella/

/kaggle/working
jaCappella  log.zip  SepACap
ballad	    meta.csv	  README.md
bossa_nova  neutral	  reggae
edm	    popular	  soulfunk
enka	    punk_rock	  test_song_list_for_vocal_ensemble_separation.txt
jazz	    README_ja.md


In [84]:
# create scp jacapella script in the data/create_scp directory
import os
import logging
from pathlib import Path
from typing import Set, Dict, List

# Set up logging for DevOps visibility and error tracking
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("SCP_Generator")

def generate_jacappella_manifests(root_dir: str, output_dir: str, test_list_path: str):
    """
    Parses the jaCappella directory structure and generates parallel .scp manifests.
    Ensures that only complete septuplets (mixture + 6 stems) are indexed.
    """
    root = Path(root_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # 1. Define the 6-stem standard required for a cappella separation 
    stems = ["alto", "bass", "lead_vocal", "soprano", "tenor", "vocal_percussion"]
    
    # 2. Security Check: Load official test song list to ensure partition integrity 
    test_songs: Set[str] = set()
    if os.path.exists(test_list_path):
        with open(test_list_path, 'r', encoding='utf-8') as f:
            test_songs = {line.strip() for line in f if line.strip()}
    else:
        logger.warning(f"Test list not found at {test_list_path}. All songs will go to Train/Valid.")

    # 3. Initialize Manifest Buffers (21 manifests total)
    manifests: Dict[str, Dict[str, List[str]]] = {
        partition: {s: [] for s in stems + ["mixture"]} 
        for partition in ["tr", "cv", "tt"]
    }

    # 4. Traversal Logic: Navigate jaCappella/{subset}/{title_in_en}/
    song_dirs = sorted([d for d in root.glob("*/*") if d.is_dir()])
    
    # Counter specifically for non-test songs to handle tr/cv split
    train_val_idx = 0 
    
    for s_dir in song_dirs:
        song_id = s_dir.name
        
        # DSA Best Practice: Strict Validation of Septuplet Completeness
        missing = [s for s in stems if not (s_dir / f"{s}.wav").exists()]
        if missing or not (s_dir / "mixture.wav").exists():
            logger.debug(f"Skipping {song_id}: Incomplete septuplet.")
            continue

        # Partitioning Logic
        if song_id in test_songs:
            partition = "tt"
        else:
            # Fixed Logic: 10% to CV, 90% to TR
            # Using a persistent counter ensures TR actually populates
            partition = "cv" if train_val_idx % 10 == 0 else "tr"
            train_val_idx += 1

        # 5. Buffer Assembly: Absolute paths for Kaggle portability
        # Add mixture
        manifests[partition]["mixture"].append(f"{song_id} {(s_dir / 'mixture.wav').absolute()}")
        
        # Add all 6 stems
        for s in stems:
            manifests[partition][s].append(f"{song_id} {(s_dir / f'{s}.wav').absolute()}")

    # 6. Atomic Write: Write all 21 manifest files (.scp)
    for part, data in manifests.items():
        for category, lines in data.items():
            file_name = out_path / f"{part}_{category}.scp"
            try:
                with open(file_name, 'w', encoding='utf-8') as f:
                    f.write("\n".join(lines) + "\n")
                logger.info(f"Generated {file_name.name} with {len(lines)} entries.")
            except IOError as e:
                logger.error(f"Failed to write manifest {file_name}: {e}")

if __name__ == '__main__':
    # Configuration matches your current Kaggle workspace
    DATA_ROOT = "/kaggle/working/jaCappella"
    SCP_OUT = "/kaggle/working/SepACap/data/scp_ss_jacappella"
    TEST_LIST = "/kaggle/working/jaCappella/test_song_list_for_vocal_ensemble_separation.txt"

    generate_jacappella_manifests(DATA_ROOT, SCP_OUT, TEST_LIST)

INFO: Generated tr_alto.scp with 36 entries.
INFO: Generated tr_bass.scp with 36 entries.
INFO: Generated tr_lead_vocal.scp with 36 entries.
INFO: Generated tr_soprano.scp with 36 entries.
INFO: Generated tr_tenor.scp with 36 entries.
INFO: Generated tr_vocal_percussion.scp with 36 entries.
INFO: Generated tr_mixture.scp with 36 entries.
INFO: Generated cv_alto.scp with 4 entries.
INFO: Generated cv_bass.scp with 4 entries.
INFO: Generated cv_lead_vocal.scp with 4 entries.
INFO: Generated cv_soprano.scp with 4 entries.
INFO: Generated cv_tenor.scp with 4 entries.
INFO: Generated cv_vocal_percussion.scp with 4 entries.
INFO: Generated cv_mixture.scp with 4 entries.
INFO: Generated tt_alto.scp with 10 entries.
INFO: Generated tt_bass.scp with 10 entries.
INFO: Generated tt_lead_vocal.scp with 10 entries.
INFO: Generated tt_soprano.scp with 10 entries.
INFO: Generated tt_tenor.scp with 10 entries.
INFO: Generated tt_vocal_percussion.scp with 10 entries.
INFO: Generated tt_mixture.scp with

In [73]:
# resample_dataset script in data/create_scp

# Mandatory Resampling: Raw jaCappella data is provided at 48kHz. 
# Before manifest generation, all files must be downsampled to 8kHz. 
# Failure to do so results in an SR Mismatch error during validation and 
# invalid spectral loss gradients during training.


import os
import librosa
import soundfile as sf
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm

def resample_file(file_path, target_sr=8000):
    """Resamples a single file and overwrites it to save space."""
    try:
        y, _ = librosa.load(file_path, sr=target_sr)
        # Overwrite the 48kHz file with the 8kHz version
        sf.write(file_path, y, target_sr)
        return True
    except Exception as e:
        print(f"Error resampling {file_path}: {e}")
        return False

def batch_resample_dataset(root_dir, target_sr=8000):
    root = Path(root_dir)
    all_wavs = list(root.rglob("*.wav"))
    
    print(f"Starting batch resampling of {len(all_wavs)} files to {target_sr}Hz...")
    
    results = Parallel(n_jobs=-1)(
        delayed(resample_file)(str(p), target_sr) for p in tqdm(all_wavs)
    )
    
    success_count = sum(results)
    print(f"Done! Successfully resampled {success_count}/{len(all_wavs)} files.")

if __name__ == '__main__':
    DATASET_ROOT = "/kaggle/working/jaCappella"
    batch_resample_dataset(DATASET_ROOT)

Starting batch resampling of 403 files to 8000Hz...


100%|██████████| 403/403 [00:07<00:00, 53.90it/s] 


Done! Successfully resampled 403/403 files.


In [85]:
# validate dataset script in data/create_scp

import os
import torch
import librosa
import logging
import numpy as np
from pathlib import Path
from tqdm import tqdm

# DevOps Visibility Configuration
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("DatasetValidator")

def validate_sepacap_standard(scp_dir: str, test_list_path: str, target_sr=8000):
    """
    Finalized validation for the jaCappella-SepACap pipeline.
    Validates the 7-file septuplet (1 mix + 6 stems) required for Power Set Augmentation.
    """
    scp_path = Path(scp_dir)
    # Standard 6-stem ensemble + 1 Mixture input
    stems = ["alto", "bass", "lead_vocal", "soprano", "tenor", "vocal_percussion", "mixture"]
    partitions = ["tr", "cv", "tt"]
    
    # 1. Leakage Prevention: Load the official experimental test split
    test_songs = set()
    if os.path.exists(test_list_path):
        with open(test_list_path, 'r', encoding='utf-8') as f:
            test_songs = {line.strip() for line in f if line.strip()}
    
    errors = 0

    for part in partitions:
        logger.info(f"--- Validating Partition: {part} ---")
        # Registry tracks song IDs across all 7 manifests to ensure parallel alignment
        song_registry = {} 
        
        for stem in stems:
            scp_file = scp_path / f"{part}_{stem}.scp"
            if not scp_file.exists():
                logger.error(f"Critical Path Error: Missing manifest -> {scp_file.name}")
                errors += 1
                continue
            
            with open(scp_file, 'r', encoding='utf-8') as f:
                # Remove empty lines that cause RuntimeError in util_dataset.py
                lines = [line.strip() for line in f if line.strip()]
                
            for line in tqdm(lines, desc=f"Checking {stem}", leave=False):
                # Robust split handles spaces in Kaggle directory paths
                parts = line.split(maxsplit=1)
                if len(parts) < 2:
                    logger.warning(f"Malformed manifest line (skipped): {line}")
                    continue
                
                song_id, audio_path = parts
                
                # Check A: Dataset Leakage (Security)
                if part != "tt" and song_id in test_songs:
                    logger.error(f"Leakage: Song '{song_id}' found in {part} but is reserved for Test set.")
                    errors += 1
                
                # Check B: File System Integrity
                if not os.path.exists(audio_path):
                    logger.error(f"Missing Audio: {audio_path}")
                    errors += 1
                    continue
                
                # Check C: Signal Processing Compatibility (Only verify once per song_id to save CPU)
                if song_id not in song_registry:
                    try:
                        # Load using header-only logic where possible for speed
                        sr = librosa.get_samplerate(audio_path)
                        
                        # 1. Sampling Rate Check (Requirement: 8000Hz)
                        if sr != target_sr:
                            logger.error(f"SR Mismatch: {audio_path} is {sr}Hz, must be {target_sr}Hz")
                            errors += 1
                        
                        # 2. Channel Check (Requirement: Mono)
                        y, _ = librosa.load(audio_path, sr=target_sr, mono=False)
                        if len(y.shape) > 1 and y.shape[0] > 1:
                            logger.error(f"Channel Error: {audio_path} is Stereo. Waveform-domain separation requires Mono.")
                            errors += 1
                        
                        samples = y.shape[-1]
                        
                        # 3. Kernel Size Floor (Requirement: >= 1024)
                        if samples < 1024:
                            logger.error(f"Audio Too Short: {audio_path} ({samples} samples). Min 1024 for STFT kernels.")
                            errors += 1
                        
                        # 4. Stride Alignment (Requirement: % 4 == 0 for SepReformer downsampling)
                        if samples % 4 != 0:
                            logger.warning(f"Stride Warning: {song_id} length {samples} is not a multiple of 4.")

                    except Exception as e:
                        logger.error(f"Corruption: Could not read {audio_path}: {e}")
                        errors += 1
                
                # Register the stem to the song_id to check parallelism later
                if song_id not in song_registry:
                    song_registry[song_id] = set()
                song_registry[song_id].add(stem)

        # Check D: Parallel Septuplet Alignment
        # This prevents the "num_samples=0" DataLoader error by ensuring every song has all 7 files
        for sid, found_stems in song_registry.items():
            if len(found_stems) != len(stems):
                missing = set(stems) - found_stems
                logger.error(f"Alignment Error: Song '{sid}' in {part} manifest is missing: {missing}")
                errors += 1

    if errors == 0:
        logger.info("✅ Dataset Validation Passed: SepACap Standard Alignment Confirmed.")
        return True
    else:
        logger.error(f"❌ Dataset Validation Failed with {errors} errors. Fix manifests before training.")
        return False

if __name__ == '__main__':
    # Configuration matches your specific Kaggle paths
    SCP_DIR = "/kaggle/working/SepACap/data/scp_ss_jacappella"
    TEST_LIST = "/kaggle/working/jaCappella/test_song_list_for_vocal_ensemble_separation.txt"
    
    validate_sepacap_standard(SCP_DIR, TEST_LIST)

INFO: --- Validating Partition: tr ---

Checking alto:   0%|          | 0/36 [00:00<?, ?it/s]

INFO: --- Validating Partition: cv ---
INFO: --- Validating Partition: tt ---
INFO: ✅ Dataset Validation Passed: SepACap Standard Alignment Confirmed.


In [80]:
import os
from pathlib import Path

# Match this to your configs.yaml 'scp_dir'
check_path = "/kaggle/working/SepACap/data/scp_ss_jacappella"
print(f"Manifests found: {os.listdir(check_path) if os.path.exists(check_path) else 'FOLDER NOT FOUND'}")

Manifests found: ['tr_vocal_percussion.scp', 'cv_bass.scp', 'tt_soprano.scp', 'tt_alto.scp', 'tr_soprano.scp', 'cv_tenor.scp', 'tr_lead_vocal.scp', 'tr_mixture.scp', 'tt_bass.scp', 'tt_tenor.scp', 'cv_lead_vocal.scp', 'cv_soprano.scp', 'tt_vocal_percussion.scp', 'cv_alto.scp', 'cv_mixture.scp', 'tr_alto.scp', 'tt_mixture.scp', 'tr_bass.scp', 'cv_vocal_percussion.scp', 'tt_lead_vocal.scp', 'tr_tenor.scp']


In [9]:
# made changes to configs.yaml
# committed to upstream repo 

!cd SepACap && git pull origin dev
#! mkdir SepACap/models/SepReformer_Base_WSJ0/log/backup && mv SepACap/models/SepReformer_Base_WSJ0/log/scratch_weights/epoch.0180.pth SepACap/models/SepReformer_Base_WSJ0/log/backup # &&git lfs install && git lfs pull
#! ls SepACap/models/SepReformer_Base_WSJ0/log/backup
#!rm -rf SepACap/data/scp_ss_jacappella

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.77 KiB | 908.00 KiB/s, done.
From https://github.com/phos-x/SepACap
 * branch            dev        -> FETCH_HEAD
   10d4faf..3a9dd20  dev        -> origin/dev
Updating 10d4faf..3a9dd20
Fast-forward
 Snake.py                                           |    21 -
 data/create_mixture_data/activlev.m                |   345 -
 data/create_mixture_data/create_wav_2speakers.m    |   191 -
 .../create_wav_2speakers_wsjmix.m                  |   153 -
 data/create_mixture_data/maxfilt.m                 |   127 -
 data/create_mixture_data/mix_2_spk_cv.txt          |  5000 ---
 data/create_mixture_data/mix_2_spk_min_cv_1        |  5000 ---
 data/create_mixture_data/mix_2_spk_min_cv_2        |  5000 ---
 data/create_mixture_data/mix_2_spk_min_cv_mix      |  5000 ---


In [ ]:
%env PYTORCH_ALLOC_CONF=expandable_segments:True
!python SepACap/run.py --model SepReformer_Base_WSJ0 --engine-mode train


env: PYTORCH_ALLOC_CONF=expandable_segments:True
2026-02-20 03:13:23.974547: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771557203.996798    1526 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771557204.003459    1526 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771557204.020836    1526 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771557204.020866    1526 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771557204.020879    1526 